# 🏠 URIS: 家居环境交互 VLM 训练 (A100 最佳效果版)

## 训练目标
微调 Qwen2.5-VL-7B-Instruct，使其能够：
1. **输出丰富有逻辑的回答** — 观察事实→分析推理→建议行动→注意事项
2. **结合上下文和用户习惯** — 利用 Object Registry 和对话历史
3. **预测用户未来行为** — 如：用户问水杯→主动问是否需要接水
4. **主动澄清歧义** — 多同类物体时优先询问而非猜测

## A100 配置
| 参数 | 值 |
|------|----|
| 模型 | Qwen2.5-VL-7B-Instruct |
| 精度 | bf16 全精度 LoRA（不量化）|
| LoRA | r=128, alpha=256 |
| Batch | 4 × 4 = 16 (effective) |
| Epochs | 5 |
| Seq Length | 8192 |
| 样本量 | 2000 |

---

## Step 1️⃣ 安装依赖

In [ ]:
!pip install -q transformers>=4.45.0 peft>=0.13.0 bitsandbytes>=0.43.0
!pip install -q accelerate>=0.34.0 datasets>=2.20.0 trl>=0.12.0
!pip install -q qwen-vl-utils Pillow requests tqdm
pip install -r requirements.txt install -q flash-attn --no-build-isolation

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 2️⃣ 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/URIS/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/URIS/train_data', exist_ok=True)
print('✅ Drive ready')

## Step 3️⃣ 配置 + System Prompt

In [ ]:
import json, random, time, copy
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any

# ===== A100 最佳配置 =====
BASE_MODEL = 'Qwen/Qwen2.5-VL-7B-Instruct'
USE_4BIT = True         # 40GB A100 必须        # A100 不量化
LORA_R = 128
LORA_ALPHA = 256
LORA_DROPOUT = 0.05
LORA_TARGETS = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
NUM_EPOCHS = 5
BATCH_SIZE = 1          # OOM 保险          # A100 80GB=4, 40GB=2
GRAD_ACCUM = 16         # effective batch=16          # effective batch = 16
LR = 5e-5
MAX_SEQ_LEN = 1024      # 40GB 安全上限
NUM_SAMPLES = 2000
VAL_RATIO = 0.1
SEED = 42
OUTPUT_DIR = '/content/drive/MyDrive/URIS/checkpoints/Qwen2.5-VL-URIS-Predictive-A100'
DATA_DIR = '/content/drive/MyDrive/URIS/train_data'

# ===== System Prompt =====
SYSTEM_PROMPT = """你是 URIS，一个专业的家居环境多模态交互助手。你的核心能力包括：

【回答原则】
1. 先给出直接结论，再展开详细分析和依据
2. 回答必须丰富、有层次、有逻辑，包含：观察事实→分析推理→建议行动→注意事项
3. 严格区分"我观察到的"和"我推断的"，标注置信度
4. 结合对话上下文和用户历史行为模式，给出个性化回答

【行为预测机制 - 核心能力】
你必须根据用户的当前行为和上下文，主动预测用户的下一步需求：
- 用户询问水杯 → 预测："需要我帮你确认杯子里是否有水吗？"
- 用户找钥匙 → 预测："你是准备出门吗？需要检查随身物品吗？"
- 用户问桌面物品 → 预测："需要我帮你整理桌面吗？"
- 用户找眼镜 → 预测："你是准备看书吗？记得注意用眼时间。"

【输出格式】
1. 面向用户的自然语言回复（丰富、友好、有预测性建议）
2. 结构化 JSON 分析块，必须包含 predicted_next_action 和 proactive_suggestion 字段
"""

print(f'✅ Config: {BASE_MODEL}')
print(f'   LoRA r={LORA_R}, alpha={LORA_ALPHA}')
print(f'   Batch={BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM}, Epochs={NUM_EPOCHS}')
print(f'   Samples={NUM_SAMPLES}, SeqLen={MAX_SEQ_LEN}')
print(f'   4-bit={USE_4BIT} (bf16 full precision LoRA)')

## Step 4️⃣ 合成训练数据（2000 条）

In [ ]:
# ===== 场景 + 行为预测映射 =====
SCENES = {
    'kitchen':     {'name':'厨房','objs':['杯子','碗','盘子','筷子','锅','水壶','微波炉','调料瓶','砧板','抹布']},
    'living_room': {'name':'客厅','objs':['遥控器','杯子','书','手机','充电器','抱枕','花瓶','纸巾盒','眼镜']},
    'bedroom':     {'name':'卧室','objs':['手机','闹钟','眼镜','书','水杯','台灯','充电线','钥匙','钱包','耳机']},
    'study':       {'name':'书房','objs':['电脑','键盘','鼠标','笔','笔记本','文件夹','台灯','水杯','手机','耳机']},
    'entrance':    {'name':'玄关','objs':['钥匙','鞋','伞','包','帽子','手套','口罩','外套','快递']},
}

BEHAVIORS = [
    {'kw':['水杯','杯子','喝水'],'queries':['帮我看看桌上的杯子','我的水杯在哪里？','杯子里还有水吗？','那个杯子是我的吗？'],
     'pred':['用户可能需要喝水或补充水分','用户可能在寻找自己的专属水杯'],
     'sug':['需要我帮你确认杯子里是否还有水吗？如果需要补水，厨房水壶里应该还有热水。',
            '看起来你在找水杯，需要我提醒你定时喝水吗？保持充足的水分很重要。'],
     'intent':'find_drink'},
    {'kw':['钥匙','出门'],'queries':['我的钥匙放在哪了？','帮我看看钥匙在不在桌上','钥匙在哪个位置？'],
     'pred':['用户可能准备出门','用户可能在做出门前的检查'],
     'sug':['你是准备出门吗？除了钥匙，需要我帮你检查手机、钱包等随身物品是否都在吗？',
            '钥匙找到了。建议出门前确认：手机、钱包、钥匙三件套是否齐全？今天天气预报显示可能下雨，要不要带伞？'],
     'intent':'prepare_to_leave'},
    {'kw':['整理','乱','桌面'],'queries':['桌面上都有什么东西？','帮我看看桌面需不需要整理','这些东西怎么归类比较好？'],
     'pred':['用户想整理桌面或房间','用户可能在准备工作或学习环境'],
     'sug':['我可以帮你分析桌面物品的分类归位建议：常用物品放近处、不常用的收纳起来。需要我给出具体整理方案吗？',
            '桌面看起来有些凌乱。建议按使用频率分三区：高频区（正前方）、中频区（侧面）、低频区（抽屉）。要我详细说明吗？'],
     'intent':'organize_space'},
    {'kw':['手机','充电'],'queries':['我的手机在哪？','手机放在哪里了？','充电器在旁边吗？'],
     'pred':['用户可能需要使用手机','用户可能需要给手机充电'],
     'sug':['手机找到了。需要我确认附近是否有充电器？如果电量低的话可以顺便充电。',
            '你的手机在那边。对了，需要我帮你检查一下充电线是否也在附近吗？'],
     'intent':'find_device'},
    {'kw':['书','学习','笔'],'queries':['书桌上的书是哪一本？','我的笔记本在哪？','帮我看看学习资料在不在'],
     'pred':['用户准备开始学习或工作','用户在准备学习环境'],
     'sug':['你是准备开始学习吗？需要我帮你确认学习区域是否整洁、光线是否充足吗？良好的学习环境能提升效率。',
            '学习资料找到了。需要我帮你规划一下学习区域的物品摆放吗？建议把常用参考书放在触手可及的位置。'],
     'intent':'prepare_study'},
    {'kw':['眼镜','看不清'],'queries':['我的眼镜在哪？','帮我找一下眼镜','眼镜放桌上了吗？'],
     'pred':['用户需要眼镜来阅读或使用电子设备'],
     'sug':['眼镜找到了。你是准备看书或用电脑吗？记得注意用眼时间，建议每40分钟休息一次眼睛。'],
     'intent':'find_accessory'},
    {'kw':['吃','食物','饿'],'queries':['厨房台面上有什么？','有没有可以吃的东西','冰箱旁边有什么食物？'],
     'pred':['用户可能饿了想找食物','用户可能在计划准备餐食'],
     'sug':['你是不是饿了？需要我帮你看看冰箱附近还有什么可用的食材吗？',
            '看起来你在找吃的。如果需要做饭，我可以根据台面上现有的食材帮你建议简单菜谱。'],
     'intent':'find_food'},
]

def make_dets(objs, n=3):
    sel = random.sample(objs, min(n, len(objs)))
    return [{'label':o,'confidence':round(random.uniform(0.65,0.98),3),
             'bbox':[random.randint(50,300),random.randint(50,300),random.randint(301,600),random.randint(301,600)],
             'center_norm':[round(random.uniform(0.1,0.9),3),round(random.uniform(0.1,0.9),3)],
             'obj_id':f'obj-{i+1:04d}'} for i,o in enumerate(sel)]

def make_reg(dets):
    return [{'obj_id':d['obj_id'],'label':d['label'],'center_norm':d['center_norm'],
             'confidence':d['confidence'],'status':'visible',
             'seen_count':random.randint(1,20),'mention_count':random.randint(0,5)} for d in dets]

def side(x): return '左侧' if x < 0.5 else '右侧'

# ===== 合成 =====
random.seed(SEED)
samples = []

for _ in range(NUM_SAMPLES):
    sk = random.choice(list(SCENES.keys()))
    sc = SCENES[sk]
    bh = random.choice(BEHAVIORS)
    q = random.choice(bh['queries'])
    dets = make_dets(sc['objs'], random.randint(2,5))
    # ensure trigger object
    for o in sc['objs']:
        if any(k in o for k in bh['kw']):
            if not any(d['label']==o for d in dets): dets[0]['label']=o
            break
    reg = make_reg(dets)
    obs = [d['label'] for d in dets]
    pred = random.choice(bh['pred'])
    sug = random.choice(bh['sug'])

    resp = (f"好的，让我帮你看看{sc['name']}的情况。\n\n"
            f"**【观察结果】**\n我在当前画面中检测到：{'、'.join(obs)}。\n\n")
    if len(obs) > 1:
        resp += (f"**【空间分析】**\n{obs[0]}在画面{side(dets[0]['center_norm'][0])}，"
                 f"{obs[1]}在{side(dets[1]['center_norm'][0])}。\n\n")
    resp += (f"**【分析与建议】**\n针对你的问题「{q}」，{pred}。\n\n"
             f"**【贴心预测】** 💡\n{sug}")

    analysis = {
        'intent':bh['intent'],'user_goal':q,'observed_objects':obs,
        'spatial_relations':[f"{d['label']}在画面{side(d['center_norm'][0])}" for d in dets[:3]],
        'scene_summary':f"{sc['name']}，检测到{len(obs)}个物体：{'、'.join(obs)}",
        'recommendation_steps':['确认目标物体位置和状态','根据用户需求提供建议','预测后续需求并主动提示'],
        'predicted_next_action':pred,'proactive_suggestion':sug,
        'clarification_needed':False,'clarifying_question':'',
        'confidence':round(random.uniform(0.7,0.92),2),
        'evidence_basis':['camera_frame','yolo_detection','object_registry','behavior_pattern'],
        'limitations':['基于2D图像分析，深度信息有限'],
    }

    ctx = json.dumps({'scene_summary':f"{sc['name']}，检测到{len(dets)}个物体",
                      'detections':dets,'object_registry':reg}, ensure_ascii=False, indent=2)
    user_msg = (f"请基于以下实时场景信息进行交互分析与建议。\n"
                f"user_query: {q}\ncontext_json:\n{ctx}")
    target = {'user_response':resp,'analysis_json':analysis}
    asst_msg = f"{resp}\n\n```json\n{json.dumps(target, ensure_ascii=False, indent=2)}\n```"

    samples.append({'messages':[
        {'role':'system','content':SYSTEM_PROMPT},
        {'role':'user','content':user_msg},
        {'role':'assistant','content':asst_msg},
    ],'task_type':bh['intent'],'scene_id':sk})

random.shuffle(samples)
val_n = max(1, int(len(samples)*VAL_RATIO))
train_data, val_data = samples[val_n:], samples[:val_n]

Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
with open(f'{DATA_DIR}/train.json','w') as f: json.dump(train_data,f,ensure_ascii=False,indent=2)
with open(f'{DATA_DIR}/val.json','w') as f: json.dump(val_data,f,ensure_ascii=False,indent=2)

print(f'✅ 数据合成完成: 训练={len(train_data)}, 验证={val_n}')

## Step 5️⃣ 加载模型 (bf16 全精度)

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig
from transformers import Qwen2_5_VLForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# A100 显存检测 + 自动降级
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB)')
if vram_gb < 35:
    print('⚠️ VRAM < 35GB, 自动切换 4-bit 量化')
    USE_4BIT = True
    BATCH_SIZE = 2

bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

print(f'Loading {BASE_MODEL} ({"4-bit" if USE_4BIT else "bf16"})...')
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True,
    dtype=torch.bfloat16, attn_implementation='flash_attention_2')
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

# Gradient checkpointing
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
print('✅ Model loaded + gradient checkpointing enabled')

## Step 6️⃣ 配置 LoRA (r=128)

In [ ]:
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, bias='none', task_type='CAUSAL_LM')

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ LoRA configured')

## Step 7️⃣ 构建 Dataset

In [ ]:
from datasets import Dataset

def tokenize_fn(example):
    text = processor.tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False)
    enc = processor.tokenizer(text, truncation=True, max_length=MAX_SEQ_LEN,
                              padding='max_length', return_tensors='pt')
    enc['labels'] = enc['input_ids'].clone()
    return {k: v.squeeze(0) for k, v in enc.items()}

train_ds = Dataset.from_list(train_data).map(
    tokenize_fn, remove_columns=['messages','task_type','scene_id'])
val_ds = Dataset.from_list(val_data).map(
    tokenize_fn, remove_columns=['messages','task_type','scene_id'])
print(f'✅ Dataset: train={len(train_ds)}, val={len(val_ds)}')

## Step 8️⃣ 开始训练 🚀

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from transformers import Trainer, TrainingArguments

# warmup_steps 替代 warmup_ratio
total_steps = (len(train_ds) // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
warmup = int(total_steps * 0.06)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_steps=warmup,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim='adamw_torch_fused',
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    save_total_limit=5,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor.tokenizer,
)

eff = BATCH_SIZE * GRAD_ACCUM
print(f'🚀 Training: {NUM_EPOCHS} epochs, batch={BATCH_SIZE}x{GRAD_ACCUM}={eff}, lr={LR}, warmup={warmup}')
trainer.train()
print('✅ Training complete!')

## Step 9️⃣ 保存 Adapter

In [ ]:
save_path = f'{OUTPUT_DIR}/final-adapter'
model.save_pretrained(save_path)
processor.save_pretrained(save_path)
print(f'✅ Saved to: {save_path}')

## Step 🔟 推理测试

In [ ]:
model.eval()

tests = [
    ('帮我看看桌上的水杯', '书房，检测到 水杯、键盘、鼠标', '预测是否需要喝水'),
    ('我的钥匙在哪？', '玄关，检测到 钥匙、鞋、包', '预测准备出门'),
    ('那个杯子是什么颜色的', '厨房，检测到 白色杯子(左)、蓝色杯子(右)', '触发澄清'),
]

for i, (q, s, exp) in enumerate(tests):
    msgs = [{'role':'system','content':SYSTEM_PROMPT},
            {'role':'user','content':f'user_query: {q}\nscene_summary: {s}'}]
    text = processor.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor.tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.9, do_sample=True)
    resp = processor.tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n{"="*60}\nTest {i+1}: {q}\nExpected: {exp}\nResponse:\n{resp[:600]}')